# Experiment 4: Optuna Hyperparameter Search — MLP on FashionMNIST

## HP Discovery + Bias Sweep — End-to-End

| Step | Description | What it does | Import path |
|------|-------------|--------------|-------------|
| 1 | Setup | Import libraries, configure paths, detect device | `src/train_utils.py`, `src/eval_utils.py` |
| 2 | Load Dataset | Persistent 54k/6k split from 60k train set only; test set untouched | --- |
| 3 | Define Model Builder | Dynamic MLP with searchable HPs + BatchNorm | — |
| 4 | Define Objective | Optuna objective with PR-AUC pruning | `src/train_utils.py`, `src/eval_utils.py` |
| 5 | Run Hyperparameter Search | 30-trial TPE search, 30 epochs/trial, Median pruner | `optuna` |
| 6 | Report Best Trial | Print best config, save all_trials.csv | — |
| 7 | Retrain Best Config | 30 epochs, best-checkpoint, dynamic batch_size | `src/train_utils.py` |
| 8 | Evaluate Best Model | Test accuracy, confusion matrix, per-class metrics | `src/eval_utils.py` |
| 9 | Logit Bias Sweep | Sweep logit bias for Shirt class trade-off | — |
| 10 | Cross-Experiment Comparison | Accuracy / Shirt TPR / Precision vs Phase 1–3 | — |

---




In [1]:
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import optuna
from optuna.trial import TrialState
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from pathlib import Path

def _find_root(marker="src", max_up=3):
    p = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.getcwd())
PROJ_ROOT = _find_root()
sys.path.insert(0, PROJ_ROOT)

from src.train_utils import train_one_epoch
from src.eval_utils import (
    get_all_probas_and_labels, compute_pr_auc_scores, evaluate_detailed,
    compute_roc_auc_scores
)

DEVICE = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
print(f'Device: {DEVICE}')

OUT_DIR = os.path.join(PROJ_ROOT, 'outputs', 'error_analysis', 'MLP', 'phase4_optuna')
DATA_DIR = os.path.join(PROJ_ROOT, 'data')
os.makedirs(OUT_DIR, exist_ok=True)

# Constants fixed from Phase 4 DB analysis (importance <= 0.05)
N_TRIALS = 20
EPOCHS_PER_TRIAL = 30
N_WARMUP_STEPS = 5
BATCH_SIZE = 512
FIXED_TEST_BATCH = 512
WEIGHT_DECAY = 1e-5
FIXED_OPTIMIZER = 'Adam'
FIXED_ACTIVATION = 'GELU'

CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(f'OUT_DIR: {OUT_DIR}')
print(f'Trials: {N_TRIALS}  Epochs/trial: {EPOCHS_PER_TRIAL}  Batch: {BATCH_SIZE}')

Device: cuda
OUT_DIR: c:\document\Study documents\Deeplearning_Course\outputs\error_analysis\MLP\phase4_optuna
Trials: 30  Epochs/trial: 30


## Dataset — train-only 54k/6k split (test set never touched during HP search)

Splits only the original 60k train set into 54k train + 6k val. The 10k test set is **never seen** during HP tuning.
Indices saved to `splits/phase4_train_indices.json` and `splits/phase4_val_indices.json` for reproducibility.


In [2]:
from pathlib import Path

# —— 60k train set only → train (54k) + val (6k). Test set (10k) never seen during HP search ——
_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
_train_full = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=False, transform=_tf)
_test_full = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=False, transform=_tf)

# Load or create persistent split (indices into the 60k train set only)
_split_dir = Path(PROJ_ROOT) / 'splits'
_train_idx_file = _split_dir / 'phase4_train_indices.json'
_val_idx_file = _split_dir / 'phase4_val_indices.json'

TRAIN_N = 54000
if _train_idx_file.exists() and _val_idx_file.exists():
    train_idx = json.load(open(_train_idx_file))
    val_idx = json.load(open(_val_idx_file))
else:
    np.random.seed(42)
    indices = np.random.permutation(len(_train_full))
    train_idx = indices[:TRAIN_N].tolist()
    val_idx = indices[TRAIN_N:].tolist()
    json.dump(train_idx, open(_train_idx_file, 'w'))
    json.dump(val_idx, open(_val_idx_file, 'w'))

_train_subset = Subset(_train_full, train_idx)   # 54k
_val_subset   = Subset(_train_full, val_idx)     # 6k

# Create DataLoaders
TRAIN_LOADER = DataLoader(_train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
VAL_LOADER   = DataLoader(_val_subset, batch_size=FIXED_TEST_BATCH, shuffle=False,
                          num_workers=0, pin_memory=True)
TEST_LOADER  = DataLoader(_test_full, batch_size=FIXED_TEST_BATCH, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f'Train: {len(_train_subset)}  Val: {len(_val_subset)}  Test: {len(_test_full)}')


Train: 215 batches  Val: 10 batches  Test: 20 batches


## Reduced MLP builder

Search space refined from Phase 4 DB analysis: 1–3 hidden layers (was 1–4),
64–1024 units (log-uniform), dropout 0.2–0.5, BatchNorm before/after activation.
Fixed: optimizer=Adam, activation=GELU, weight_decay=1e-5.


In [3]:
def build_mlp(trial):
    n_layers = trial.suggest_int('n_layers', 1, 3)
    units = []
    for i in range(n_layers):
        units.append(trial.suggest_int(f'units_{i}', 64, 1024, log=True))
    dropout = trial.suggest_float('dropout', 0.2, 0.5, step=0.05)
    batch_norm = trial.suggest_categorical('batch_norm', ['before_act', 'after_act'])
    layers = [nn.Flatten()]
    in_dim = 784
    for i, out_dim in enumerate(units):
        layers.append(nn.Linear(in_dim, out_dim))
        if batch_norm == 'before_act':
            layers.append(nn.BatchNorm1d(out_dim))
        layers.append(nn.GELU())
        if batch_norm == 'after_act':
            layers.append(nn.BatchNorm1d(out_dim))
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        in_dim = out_dim
    layers.append(nn.Linear(in_dim, 10))
    return nn.Sequential(*layers)

## Optuna search — objective function

**Pruning metric**: macro PR-AUC (computed on validation set every epoch).
**Reduced search space** (from Phase 4 DB analysis):
  - Fixed: optimizer=Adam, activation=GELU, weight_decay=1e-5, batch_size=512
  - Dropped: SGD, scheduler=none, batch_norm=none, ELU, LeakyReLU, 4-layer nets
  - Narrowed: lr [1e-4, 2e-3], dropout [0.2, 0.5], n_layers [1, 3]
**Improved pruning**: n_warmup_steps=5 (was 3) — reduces false-positive pruning.

In [4]:
def objective(trial):
    torch.manual_seed(42)
    if DEVICE.startswith('cuda'):
        torch.cuda.manual_seed_all(42)

    # ── Sample HPs (reduced space) ──
    lr = trial.suggest_float('lr', 1e-4, 2e-3, log=True)
    scheduler_on = trial.suggest_categorical('scheduler', ['step', 'cosine'])

    model = build_mlp(trial).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    criterion = nn.CrossEntropyLoss()
    if scheduler_on == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PER_TRIAL)
    else:
        step_size = trial.suggest_int('step_size', 5, 10)
        gamma = trial.suggest_float('gamma', 0.1, 0.5)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    # ── Train with intermediate pruning ──
    train_loader = DataLoader(_train_subset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=0, pin_memory=True)
    for epoch in range(EPOCHS_PER_TRIAL):
        loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        scheduler.step()
        probas, labels = get_all_probas_and_labels(model, VAL_LOADER, DEVICE, 10)
        pr_scores = compute_pr_auc_scores(probas, labels, model_name='_trial')
        val_pr = pr_scores['macro']
        trial.report(val_pr, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    # ── Final validation accuracy ──
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in VAL_LOADER:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = model(images).argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    trial.set_user_attr('val_acc', 100 * correct / total)

    return val_pr

## Run the study (refined search, improved pruning)

Pruner: MedianPruner with n_warmup_steps=5 (was 3) to avoid premature pruning.
Sampler: TPESampler(seed=42). N_TRIALS=20 (reduced due to tighter search space).


In [5]:
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5, n_warmup_steps=N_WARMUP_STEPS, interval_steps=1
    ),
    study_name='mlp_fashionmnist_v2',
    storage=f'sqlite:///{OUT_DIR}/optuna_study.db',
)
study.optimize(objective, n_trials=N_TRIALS, timeout=None, show_progress_bar=True)

[I 2026-07-27 22:21:06,189] A new study created in RDB with name: mlp_fashionmnist


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-07-27 22:27:25,231] Trial 0 finished with value: 0.9528556266163688 and parameters: {'batch_size': 128, 'lr': 0.0002051338263087451, 'weight_decay': 2.9375384576328313e-06, 'optimizer': 'AdamW', 'scheduler': 'step', 'n_layers': 4, 'units_0': 99, 'units_1': 93, 'units_2': 93, 'units_3': 120, 'activation': 'GELU', 'dropout': 0.05, 'batch_norm': 'after_act', 'step_size': 9, 'gamma': 0.1798695128633439}. Best is trial 0 with value: 0.9528556266163688.
[I 2026-07-27 22:32:30,720] Trial 1 finished with value: 0.9493194775192407 and parameters: {'batch_size': 512, 'lr': 0.00021930485556643703, 'weight_decay': 1.5673095467235414e-06, 'optimizer': 'AdamW', 'scheduler': 'step', 'n_layers': 2, 'units_0': 164, 'units_1': 358, 'activation': 'LeakyReLU', 'dropout': 0.15000000000000002, 'batch_norm': 'before_act', 'step_size': 10, 'gamma': 0.4100531293444458}. Best is trial 0 with value: 0.9528556266163688.
[I 2026-07-27 22:39:14,352] Trial 2 finished with value: 0.9335302124223037 and parame

In [6]:
pruned = sum(1 for t in study.trials if t.state == TrialState.PRUNED)
complete = sum(1 for t in study.trials if t.state == TrialState.COMPLETE)
print(f'Study completed: {complete} complete, {pruned} pruned')

best = study.best_trial
print(f'\n{"="*70}')
print(f'Best trial: #{best.number}')
print(f'Val macro PR-AUC: {best.value:.6f}')
print(f'Val accuracy: {best.user_attrs.get("val_acc", "N/A"):.2f}%')
print(f'Params:')
for k, v in best.params.items():
    print(f'  {k}: {v}')

# ── Save raw data ──
df = study.trials_dataframe()
df.to_csv(os.path.join(OUT_DIR, 'all_trials.csv'), index=False)
df_complete = df[df['state'] == 'COMPLETE']
df_complete.to_csv(os.path.join(OUT_DIR, 'complete_trials.csv'), index=False)

# ── Identify dead zones ──
print(f'\n{"="*70}')
print('DEAD ZONES (100% pruned):')
print(f'{"="*70}')
for col in df.columns:
    if not col.startswith('params_'):
        continue
    # Filter to rows where state is meaningful
    sub = df[df['state'].isin(['COMPLETE', 'PRUNED'])]
    ct = sub.groupby([col, 'state']).size().unstack(fill_value=0)
    total = ct.sum(axis=1)
    pruned_only = ct.get('PRUNED', 0) == total
    dead = pruned_only[pruned_only].index.tolist()
    if dead:
        print(f'  {col.replace("params_", "")}: {dead}')

# ── Top-5 complete trials ──
if len(df_complete) > 0:
    top5 = df_complete.sort_values('value', ascending=False).head(5)
    print(f'\n{"="*70}')
    print('TOP 5 COMPLETE TRIALS:')
    print(f'{"="*70}')
    for _, r in top5.iterrows():
        dur = r['duration']
        dur_min = dur.total_seconds() / 60 if hasattr(dur, 'total_seconds') else dur
        print(f'  #{int(r["number"]):2d}  value={r["value"]:.6f}  dur={dur_min:.1f}min')
        for k in [c for c in df.columns if c.startswith('params_')]:
            v = r.get(k)
            if not pd.isna(v):
                print(f'      {k.replace("params_", "")}={v}')
        print()

Study completed: 11 complete, 19 pruned

Best trial: #17
Val macro PR-AUC: 0.959593
Val accuracy: 90.40%
Params:
  batch_size: 512
  lr: 0.0004354209536991517
  weight_decay: 1.0835847514722128e-06
  optimizer: Adam
  scheduler: step
  n_layers: 2
  units_0: 666
  units_1: 753
  activation: GELU
  dropout: 0.30000000000000004
  batch_norm: after_act
  step_size: 6
  gamma: 0.40447932839439255


## Visualization: Parameter Importance


In [31]:
# ── Parameter importance plot ──
from optuna.visualization import plot_param_importances, plot_slice, plot_contour, plot_intermediate_values, plot_parallel_coordinate

fig_imp = plot_param_importances(study)
fig_imp.write_html(os.path.join(OUT_DIR, 'param_importance.html'))
fig_imp.show()


## Visualization: Slice Plot


In [32]:
fig_slice = plot_slice(study)
fig_slice.write_html(os.path.join(OUT_DIR, 'slice_plot.html'))
fig_slice.show()

## Visualization: Contour Plot (lr × dropout × n_layers)


In [33]:
# Plot the top-3 most important HPs as contour
fig_contour = plot_contour(study, params=['lr', 'dropout', 'n_layers'])
fig_contour.write_html(os.path.join(OUT_DIR, 'contour_plot.html'))
fig_contour.show()

## Visualization: Intermediate Values (per-trial learning curves)


In [37]:
# ── Intermediate values (per-trial learning curves) ──
fig_inter = plot_intermediate_values(study)
fig_inter.write_html(os.path.join(OUT_DIR, 'intermediate_values.html'))
fig_inter.show()

## Visualization: Parallel Coordinate Plot


In [35]:
# ── Parallel coordinate plot ──
fig_par = plot_parallel_coordinate(study)
fig_par.write_html(os.path.join(OUT_DIR, 'parallel_coordinate.html'))
fig_par.show()

## Best config — retrain 30 epochs with best-checkpoint early stopping

Reconstructs from the best HP configuration, trains 30 epochs tracking validation accuracy,
and restores the best checkpoint before final test evaluation.


In [ ]:
# ── Rebuild model from best params ──
class FixedTrial:
    def __init__(self, params):
        self._params = params
    def suggest_int(self, name, low, high, log=False):
        return self._params[name]
    def suggest_float(self, name, low, high, log=False, step=None):
        return self._params[name]
    def suggest_categorical(self, name, choices):
        return self._params[name]

trial_stub = FixedTrial(best.params)
model = build_mlp(trial_stub).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {n_params:,}')

optimizer = optim.Adam(model.parameters(), lr=best.params['lr'], weight_decay=WEIGHT_DECAY)

scheduler = None
if best.params['scheduler'] == 'cosine':
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
elif best.params['scheduler'] == 'step':
    scheduler = optim.lr_scheduler.StepLR(
        optimizer, step_size=best.params['step_size'],
        gamma=best.params['gamma'])

criterion = nn.CrossEntropyLoss()

# ── Train 30 epochs with best-val checkpoint ──
train_loader = DataLoader(_train_subset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0, pin_memory=True)
print(f'\nTraining best config for 30 epochs...')
train_losses = []
val_losses = []
best_val_acc = -1.0
best_epoch = -1
for epoch in range(30):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    train_losses.append(loss)
    if scheduler:
        scheduler.step()

    model.eval()
    correct = total = 0
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in VAL_LOADER:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            val_loss += criterion(logits, labels).item() * labels.size(0)
            preds = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    val_loss /= total
    val_losses.append(val_loss)
    val_acc_epoch = 100 * correct / total

    if val_acc_epoch > best_val_acc:
        best_val_acc = val_acc_epoch
        best_epoch = epoch
        torch.save(model.state_dict(), os.path.join(OUT_DIR, 'best_checkpoint.pth'))

    if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == 29:
        print(f'  Epoch [{epoch+1:2d}/30] Loss: {loss:.4f}  ValLoss: {val_loss:.4f}  ValAcc: {val_acc_epoch:.2f}%')

print(f'  Best checkpoint at epoch {best_epoch+1}: val_acc={best_val_acc:.2f}%')
model.load_state_dict(torch.load(os.path.join(OUT_DIR, 'best_checkpoint.pth')))
model.to(DEVICE)

# Save training curves
np.savetxt(os.path.join(OUT_DIR, 'train_losses_best.txt'), train_losses)
np.savetxt(os.path.join(OUT_DIR, 'val_losses_best.txt'), val_losses)

Params: 1,035,439

Training best config for 30 epochs...
  Epoch [1/30] Loss: 0.5031  ValAcc: 86.08%
  Epoch [5/30] Loss: 0.2872  ValAcc: 88.88%
  Epoch [10/30] Loss: 0.2039  ValAcc: 89.62%
  Epoch [15/30] Loss: 0.1599  ValAcc: 90.08%
  Epoch [20/30] Loss: 0.1356  ValAcc: 90.26%


In [ ]:
# ── Comprehensive test evaluation ──
probas, labels = get_all_probas_and_labels(model, TEST_LOADER, DEVICE, 10)

# PR-AUC
pr_scores = compute_pr_auc_scores(probas, labels, model_name='optuna_best')
pr_macro = pr_scores['macro']
print(f'Test macro PR-AUC: {pr_macro:.6f}')

# ROC-AUC
roc_scores = compute_roc_auc_scores(probas, labels, model_name='optuna_best')
roc_macro = roc_scores['macro']
print(f'Test macro ROC-AUC: {roc_macro:.6f}')

# Detailed evaluation (accuracy, per-class metrics, confusion matrix)
test_acc, cm, per_class = evaluate_detailed(
    model, TEST_LOADER, DEVICE, CLASS_NAMES, model_name='optuna_best')

# ── Per-class ROC-AUC breakdown ──
labels_np = labels.cpu().numpy()
probas_np = probas.cpu().numpy()
roc_per_class = {}
for i in range(10):
    y_true = (labels_np == i).astype(int)
    y_score = probas_np[:, i]
    roc_per_class[CLASS_NAMES[i]] = roc_auc_score(y_true, y_score)

print(f'\n{"="*70}')
print('PER-CLASS ROC-AUC:')
print(f'{"="*70}')
for name, score in roc_per_class.items():
    print(f'  {name:<15s} {score:.4f}')

# ── Save comprehensive results ──
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w') as f:
    json.dump({
        'number': best.number,
        'val_macro_pr_auc': best.value,
        'test_macro_pr_auc': pr_macro,
        'test_macro_roc_auc': roc_macro,
        'val_accuracy': best.user_attrs.get('val_acc', None),
        'test_accuracy': test_acc,
        'params_count': n_params,
        'best_val_epoch': best_epoch,
        'best_val_acc_at_ckpt': best_val_acc,
        'params': best.params,
        'per_class_roc_auc': roc_per_class,
    }, f, indent=2)

torch.save(model.state_dict(), os.path.join(OUT_DIR, 'model_weights.pth'))
print(f'\nAll results saved to {OUT_DIR}/')

  Test Accuracy: 90.00%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8390     0.0177     0.8407
  Trouser             0.9800     0.0004     0.9959
  Pullover            0.8430     0.0207     0.8192
  Dress               0.9080     0.0111     0.9008
  Coat                0.8550     0.0184     0.8374
  Sandal              0.9640     0.0028     0.9747
  Shirt               0.7120     0.0266     0.7487
  Sneaker             0.9620     0.0062     0.9450
  Bag                 0.9760     0.0030     0.9731
  Ankle boot          0.9610     0.0042     0.9620

All results saved to c:\document\Study documents\Deeplearning_Course\outputs\error_analysis\MLP\phase4_optuna/


## Logit bias sweep (same as Phase 1 A3)


In [ ]:
SHIRT_IDX = 6
BIASES = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
sweep = []

model.eval()
with torch.no_grad():
    for bias in BIASES:
        all_p, all_l = [], []
        sh_tp = sh_fp = sh_fn = 0
        for inputs, lbls in TEST_LOADER:
            inputs, lbls = inputs.to(DEVICE), lbls.to(DEVICE)
            logits = model(inputs)
            logits[:, SHIRT_IDX] += bias
            preds = logits.argmax(dim=1)
            all_p.extend(preds.cpu().numpy()); all_l.extend(lbls.cpu().numpy())
            for true, pred in zip(lbls.cpu().numpy(), preds.cpu().numpy()):
                if pred == SHIRT_IDX and true == SHIRT_IDX: sh_tp += 1
                if pred == SHIRT_IDX and true != SHIRT_IDX: sh_fp += 1
                if pred != SHIRT_IDX and true == SHIRT_IDX: sh_fn += 1
        acc = accuracy_score(all_l, all_p)
        sweep.append({'bias': bias, 'acc': round(acc*100, 2),
                       'tpr': round(sh_tp/(sh_tp+sh_fn+1e-8), 4),
                       'prec': round(sh_tp/(sh_tp+sh_fp+1e-8), 4)})
        print(f'bias={bias:+.1f}  acc={acc*100:.2f}%  '
              f'Shirt TPR={sh_tp/(sh_tp+sh_fn+1e-8):.4f}  '
              f'Prec={sh_tp/(sh_tp+sh_fp+1e-8):.4f}')

with open(os.path.join(OUT_DIR, 'bias_sweep_results.txt'), 'w') as f:
    f.write(f'{"Bias":>6} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}\n'
            + '-' * 32 + '\n')
    for r in sweep:
        f.write(f'{r["bias"]:>+5.1f} {r["acc"]:>7.2f} '
                f'{r["tpr"]:>9.4f} {r["prec"]:>10.4f}\n')


bias=-1.0  acc=89.93%  Shirt TPR=0.6420  Prec=0.8005
bias=-0.5  acc=90.00%  Shirt TPR=0.6770  Prec=0.7764
bias=+0.0  acc=90.00%  Shirt TPR=0.7120  Prec=0.7487
bias=+0.5  acc=89.87%  Shirt TPR=0.7440  Prec=0.7120
bias=+1.0  acc=89.80%  Shirt TPR=0.7790  Prec=0.6876
bias=+1.5  acc=89.43%  Shirt TPR=0.8120  Prec=0.6501
bias=+2.0  acc=89.05%  Shirt TPR=0.8450  Prec=0.6177


## Comparison vs Phase 1 baseline, Wider, and Deeper


In [ ]:
bt = max(sweep, key=lambda r: r['acc'] + r['tpr'] * 100)
z = [r for r in sweep if r['bias'] == 0.0][0]

print(f'{"Config":<30} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}')
print('-' * 56)
print(f'{"Phase 1 baseline":<30} {"90.08":>7} {"0.7070":>9} {"0.7505":>10}')
print(f'{"Exp 2 Wider":<30} {"90.26":>7} {"0.7280":>9} {"0.7599":>10}')
print(f'{"Exp 3 Deeper":<30} {"90.13":>7} {"0.7140":>9} {"0.7645":>10}')
print(f'{"Exp 4 optuna bias=0":<30} {z["acc"]:>7.2f} {z["tpr"]:>9.4f} {z["prec"]:>10.4f}')
print(f'{"Exp 4 bias="+str(bt["bias"])+" (best trade)":<30} '
      f'{bt["acc"]:>7.2f} {bt["tpr"]:>9.4f} {bt["prec"]:>10.4f}')


Config                            Acc%  ShirtTPR  ShirtPrec
--------------------------------------------------------
Phase 1 baseline                 90.08    0.7070     0.7505
Exp 2 Wider                      90.26    0.7280     0.7599
Exp 3 Deeper                     90.13    0.7140     0.7645
Exp 4 optuna bias=0              90.00    0.7120     0.7487
Exp 4 bias=2.0 (best trade)      89.05    0.8450     0.6177
